# m52 lab · Cardiac MRI for EP: from spins to "fibrosis" burden

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You compute the Larmor frequency, simulate an inversion pulse with the Bloch equations, see how gadolinium
shortens T1 and how extracellular volume follows from it, choose the inversion time for late gadolinium
enhancement (LGE), measure how much "fibrosis" a synthetic atrial wall shows under each published threshold, try
blood-pool normalization on real heart MRI, and read the evidence chain from association to trial.

| Part | What you do | Data |
| --- | --- | --- |
| A | The Larmor frequency (eq-m52-larmor) | CODATA 2022 |
| B | Inversion recovery from the Bloch equations (eq-m52-ir) | our own simulation |
| C | Gadolinium, T1 and extracellular volume (eq-m52-relaxivity, eq-m52-ecv) | published T1 values |
| D | Choosing the inversion time (eq-m52-lge) | published T1 values |
| E | Thresholds and "fibrosis" burden on a synthetic wall (eq-m52-iir) | our own phantom |
| F | Blood-pool normalization on real heart MRI (optional download) | Medical Segmentation Decathlon |
| G | Reading the evidence: association, trial, method | published results |

Values come from published sources where one exists: the proton gyromagnetic ratio (CODATA 2022), gadolinium
relaxivity and the dose example (Kellman and Arai 2012), and T1 and extracellular volume of myocardium and chronic
infarct at 1.5 T (Bauner et al. 2012). Native blood T1, hematocrit, the RF pulse, T2 and every value of the
synthetic wall are illustrative. Each part ends with **checks**. Part F downloads a public data set (several
hundred MB, CC BY-SA 4.0) once; set `EPML_SKIP_REAL_DATA=1` to skip it.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import mri
from epml.data import DataUnavailable
from epml.imaging import dice

## Part A · The Larmor frequency

Protons in a static field $B_0$ precess at $f = (\gamma/2\pi)\,B_0$ (eq-m52-larmor), with
$\gamma/2\pi$ = 42.577 MHz/T (CODATA 2022). The scanner transmits and receives at this frequency; a field
gradient makes it depend on position, which is how MRI encodes space.

In [ ]:
for b0 in (1.5, 3.0):
    print(f"{b0} T: {float(mri.larmor_mhz(b0)):.2f} MHz")
# a gradient of 10 mT/m spreads frequencies over a 300-mm field of view by:
spread_khz = float(mri.larmor_mhz(10e-3 * 0.3)) * 1e3
print(f"10 mT/m over 300 mm: {spread_khz:.0f} kHz")

In [ ]:
# Checks
assert abs(float(mri.larmor_mhz(1.5)) - 63.87) < 0.01
assert abs(spread_khz - 127.7) < 0.1

The gradient strength here is illustrative.

## Part B · Inversion recovery from the Bloch equations

An RF pulse at the Larmor frequency tips the magnetization; a 180° pulse inverts it. Afterwards $M_z$ recovers
toward $M_0$ with time constant $T_1$: $M_z(t) = M_0(1 - 2e^{-t/T_1})$, crossing zero at $T_1\ln 2$ (eq-m52-ir).
Here we integrate the full Bloch equations (rotating frame, on resonance) through a finite pulse and compare. The
pulse amplitude and $T_2$ are illustrative; $T_1$ = 379 ms is the mean of remote myocardium 10 minutes after
gadolinium at 1.5 T (Bauner et al. 2012).

In [ ]:
t1 = mri.BAUNER["myocardium"]["t1_post"]
t, mx, my, mz = mri.bloch_inversion(t1, 50.0)
plt.plot(t, mz, label="Bloch simulation")
plt.plot(t, mri.ir_mz(t, t1), "--", label="M0 (1 - 2 exp(-t/T1))")
plt.axvline(mri.null_time(t1), color="gray", ls=":", label="T1 ln 2")
plt.xlabel("time after the pulse (ms)"); plt.ylabel("Mz / M0"); plt.legend(); plt.show()
dev = np.abs(mz - mri.ir_mz(t, t1)).max()
print(f"largest difference {dev:.4f}; Mz just after the pulse {mz[0]:.4f} (T2 decay during the ~1 ms pulse)")
print(f"null at {float(mri.null_time(t1)):.0f} ms")

In [ ]:
# Checks
assert dev < 0.02
assert abs(t[np.argmin(np.abs(mz))] - mri.null_time(t1)) < 6

## Part C · Gadolinium, T1 and extracellular volume

Gadolinium agents raise the relaxation rate in proportion to their concentration:
$1/T_1 = 1/T_{1,0} + r_1[\mathrm{Gd}]$ (eq-m52-relaxivity), with $r_1$ = 4.5 L/(mmol·s) (Kellman and Arai 2012).
They stay outside cells, so a tissue's concentration is its extracellular volume fraction (ECV) times the
plasma's, and blood's is $(1 - \mathrm{Hct})$ times the plasma's. Dividing the two changes in $R_1 = 1/T_1$ gives
$\mathrm{ECV} = (1 - \mathrm{Hct})\,\Delta R_{1,\text{myo}}/\Delta R_{1,\text{blood}}$ (eq-m52-ecv). Kellman
and Arai's worked example: 0.15 mmol/kg into blood that is 8% of body weight at 1.06 kg/L, tissue ECV 25%, native
T1 950 ms.

In [ ]:
cb = mri.blood_concentration(0.15, 0.08, 1.06)
ct = 0.25 * cb
print(f"blood {cb:.2f} mmol/L, tissue {ct:.2f} mmol/L, dR1 {mri.RELAXIVITY * ct:.2f} /s, "
      f"T1 950 -> {float(mri.t1_with_gd(950.0, ct)):.0f} ms")

b = mri.BAUNER
dm = mri.delta_r1(b["myocardium"]["t1_pre"], b["myocardium"]["t1_post"])
di = mri.delta_r1(b["infarct"]["t1_pre"], b["infarct"]["t1_post"])
print(f"Bauner et al. 2012: dR1 myocardium {dm:.2f} /s, infarct {di:.2f} /s, ratio {di / dm:.2f}; "
      f"ECV ratio {b['infarct']['ecv'] / b['myocardium']['ecv']:.2f}")

# the same model gives blood, with an illustrative native T1 and hematocrit
t1_blood = mri.blood_t1_post_illustrative()
print(f"blood after contrast (illustrative native T1 {mri.BLOOD_T1_PRE_ILLUSTRATIVE:.0f} ms, Hct {mri.HCT_ILLUSTRATIVE}): "
      f"{t1_blood:.0f} ms")
back = mri.ecv(b["myocardium"]["t1_pre"], b["myocardium"]["t1_post"], mri.BLOOD_T1_PRE_ILLUSTRATIVE, t1_blood,
               mri.HCT_ILLUSTRATIVE)
print(f"ECV recovered from those four T1 values: {back:.3f}")

In [ ]:
# Checks
assert abs(float(mri.t1_with_gd(950.0, ct)) - 300) < 10
assert abs(di / dm - 2.04) < 0.01
assert abs(back - b["myocardium"]["ecv"]) < 1e-9

The infarct's change in $R_1$ is about twice the myocardium's, close to the ratio of their ECVs (Bauner et
al. 2012): scar has more space for gadolinium, so its T1 falls further. Blood ends up close to scar, which is why
scar next to the blood pool can be hard to see.

## Part D · Choosing the inversion time

In an LGE image the signal is read a time TI after the inversion. Ignoring the readout, it is
$|1 - 2e^{-\mathrm{TI}/T_1}|$ for magnitude images. Choosing TI to null healthy myocardium makes it black and
leaves scar at $|1 - 2^{1 - T_{1m}/T_{1s}}|$; the signed difference between scar and myocardium peaks a little
later, at $\mathrm{TI}^* = \ln(T_{1m}/T_{1s})/(1/T_{1s} - 1/T_{1m})$ (eq-m52-lge). Phase-sensitive reconstruction
keeps the sign, so a TI that is too short does not invert the contrast (Kellman and Arai 2012).

In [ ]:
t1m, t1s = b["myocardium"]["t1_post"], b["infarct"]["t1_post"]
ti = np.linspace(0, 700, 701)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for mag, a in ((True, ax[0]), (False, ax[1])):
    for name, t1v, style in (("myocardium", t1m, "-"), ("scar", t1s, "-"), ("blood (illustrative)", t1_blood, "--")):
        a.plot(ti, mri.lge_signal(ti, t1v, magnitude=mag), style, label=name)
    a.axvline(mri.null_time(t1m), color="gray", ls=":")
    a.set_xlabel("TI (ms)"); a.set_title("magnitude" if mag else "phase-sensitive")
ax[0].set_ylabel("signal / M0"); ax[0].legend()
plt.tight_layout(); plt.show()

ti_null, ti_best = float(mri.null_time(t1m)), float(mri.best_ti_signed(t1m, t1s))
print(f"myocardial null {ti_null:.0f} ms; scar there {mri.scar_signal_at_null(t1m, t1s):.2f}, "
      f"blood {float(mri.lge_signal(ti_null, t1_blood)):.2f}")
print(f"signed difference peaks at {ti_best:.0f} ms")
early = 150.0
print(f"at TI {early:.0f} ms (too short), magnitude: myocardium {float(mri.lge_signal(early, t1m)):.2f}, "
      f"scar {float(mri.lge_signal(early, t1s)):.2f}")

In [ ]:
# Checks
assert ti_null < ti_best < ti_null + 50
assert float(mri.lge_signal(early, t1m)) > float(mri.lge_signal(early, t1s))   # contrast inverted when TI is too short

Real sequences read out during recovery, which shortens the apparent T1 (Kellman and Arai 2012), so the
null found on a scanner is not exactly $T_1\ln 2$.

## Part E · Thresholds and "fibrosis" burden on a synthetic wall

To compare scans, wall intensity is divided by the mean blood-pool intensity: the image intensity ratio,
$\mathrm{IIR} = I_{\text{wall}}/\bar I_{\text{blood}}$ (eq-m52-iir). A threshold then calls each wall sample
fibrotic or not. Our synthetic wall has known patches covering 15% of it; its normal tissue has mean IIR 0.9 and SD
0.155, so that mean + 2 SD is 1.21, the upper normal limit measured in the right atrium at 3 T (Gunturiz-Beltrán et
al. 2022). Everything else about it is illustrative. We apply the published thresholds and compare each with the
truth and with a repeat scan of the same wall (new noise), in the spirit of Kamali et al. 2020 and Boyle et al.
2022.

In [ ]:
rng = np.random.default_rng(52)
w = mri.synthetic_wall(rng)
rep = mri.scan(rng, w["clean"])
wi, wi2 = mri.iir(w["wall"], w["blood"]), mri.iir(rep["wall"], rep["blood"])
bn, bn2 = w["blood"] / w["blood"].mean(), rep["blood"] / rep["blood"].mean()
true_burden = w["truth"].mean()

rows = []
for name, kv in mri.THRESHOLDS.items():
    t_a, t_b = mri.threshold_for(name, kv, wi, bn), mri.threshold_for(name, kv, wi2, bn2)
    rows.append((name, t_a, mri.burden(wi, t_a), dice(wi > t_a, w["truth"]), dice(wi > t_a, wi2 > t_b)))
print(f"true burden {true_burden:.0%}")
print(f"{'method':66s} {'thr':>5s} {'burden':>7s} {'Dice truth':>10s} {'Dice repeat':>11s}")
for name, t_a, bur, d_t, d_r in rows:
    print(f"{name:66s} {t_a:5.2f} {bur:7.1%} {d_t:10.2f} {d_r:11.2f}")

ths = np.linspace(0.8, 2.0, 121)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].imshow(wi, origin="lower", cmap="gray", vmin=0.3, vmax=2.0, aspect="auto")
ax[0].contour(w["truth"], levels=[0.5], colors="C1", linewidths=0.8)
ax[0].set_title("wall IIR, true patches outlined"); ax[0].set_xlabel("around the atrium"); ax[0].set_ylabel("up the wall")
ax[1].plot(ths, [mri.burden(wi, t) for t in ths])
ax[1].axhline(true_burden, color="gray", ls="--", label="true burden")
ax[1].set_xlabel("IIR threshold"); ax[1].set_ylabel("burden"); ax[1].legend()
plt.tight_layout(); plt.show()

burdens = [r[2] for r in rows]
# a common gain (coil sensitivity, scanner) cancels in the ratio
gain_ok = np.allclose(mri.iir(2.5 * w["wall"], 2.5 * w["blood"]), wi)

In [ ]:
# Checks
assert min(burdens) < true_burden < max(burdens)
assert max(burdens) / min(burdens) > 5
assert gain_ok
assert all(r[4] < 0.95 for r in rows)          # no method reproduces itself perfectly on a repeat scan

The same wall reads anywhere from a few percent to almost half "fibrotic", depending only on the threshold;
Boyle et al. 2022 found an average of 5.0% to 37.4% on real scans as the IIR threshold moved from 1.2 to 0.97. Our
numbers come from an illustrative phantom and are not a validation of any method.

## Part F · Blood-pool normalization on real heart MRI (optional download)

The Medical Segmentation Decathlon's heart task (Antonelli et al. 2021; CC BY-SA 4.0), already used in m40,
provides 20 MRI volumes with the left atrium labelled. **These are not LGE images**, so nothing here is
fibrosis. We use them only to see blood-pool normalization on real noise and coil shading: the "wall" is a 3-mm
shell outside the label, and we report how much of it exceeds each IIR threshold.

In [ ]:
try:
    import nibabel as nib
    from epml.data import fetch_msd_heart
    pairs = fetch_msd_heart()
    REAL = True
except DataUnavailable as err:
    print("Part F skipped:", err)
    REAL = False

if REAL:
    table = []
    for ip, lp in pairs[:6]:
        li = nib.load(str(lp))
        spacing = tuple(float(z) for z in li.header.get_zooms()[:3])
        lab = np.asarray(li.dataobj) > 0
        img = np.asarray(nib.load(str(ip)).dataobj, dtype=np.float32)
        img_c, lab_c = mri.crop_to_label(img, lab, spacing, 10.0)
        vals, mean_blood, n_blood = mri.shell_iir(img_c, lab_c, spacing, (0.0, 3.0))
        scaled, _, _ = mri.shell_iir(img_c / 1000.0, lab_c, spacing, (0.0, 3.0))
        table.append((Path(ip).name, n_blood, float(np.median(vals)),
                      [mri.burden(vals, t) for t in (0.97, 1.2, 1.61)], bool(np.allclose(vals, scaled))))
    print(f"{'case':22s} {'blood vox':>9s} {'median IIR':>10s}  {'>0.97':>6s} {'>1.2':>6s} {'>1.61':>6s}")
    for name, n, med, (b1, b2, b3), _ in table:
        print(f"{name:22s} {n:9d} {med:10.2f}  {b1:6.1%} {b2:6.1%} {b3:6.1%}")
    plt.hist(vals, bins=60, range=(0, 2))
    for t in (0.97, 1.2, 1.61):
        plt.axvline(t, color="gray", ls=":")
    plt.xlabel("shell intensity / blood-pool mean (last case)"); plt.ylabel("voxels"); plt.show()

In [ ]:
# Checks
if REAL:
    assert all(0 <= x <= 1 for row in table for x in row[3])
    assert all(row[4] for row in table)                    # scaling the image does not change the ratio
    assert all(row[1] > 0 for row in table)

Even without any scar, the share of the shell above a threshold varies from case to case, because the shell
mixes wall, fat, neighboring blood and noise, and because each scan has its own contrast. On LGE images the same
mechanics decide the reported burden.

## Part G · Reading the evidence: association, trial, method

| Study | Question | Design | Finding |
| --- | --- | --- | --- |
| Marrouche et al. 2014 | Does atrial fibrosis on LGE predict recurrence after AF ablation? | Multicenter cohort | Associated with recurrence (hazard ratio 1.06 per 1% fibrosis) |
| Marrouche et al. 2022 | Does ablating that fibrosis, added to PVI, reduce recurrence? | Randomized trial, 843 patients | No significant difference (43.0% vs 46.1%) |
| Akoum et al. 2024 | Did the fibrosis-guided lesions become scar? | Secondary analysis of that trial | Not uniformly, with large differences between centers |
| Boyle et al. 2022 | Do two quantification methods agree? | Single-center cohort, 102 patients | Burden depended on method and threshold; spatial overlap modest (Dice 0.49) |
| Kamali et al. 2020 | Are thresholds reproducible on repeat scans? | Cohort, 45 patients, two scans | Otsu most reproducible (Dice 71) against 3.3 SD (58) and IIR 1.61 (46) |

In [ ]:
evidence = [
    ("Marrouche 2014", "cohort", "association"),
    ("Marrouche 2022", "randomized trial", "intervention"),
    ("Akoum 2024", "secondary analysis", "mechanism"),
    ("Boyle 2022", "cohort", "method"),
    ("Kamali 2020", "cohort", "method"),
]
randomized = [e for e in evidence if e[1] == "randomized trial"]
print("Randomized tests of acting on MRI fibrosis in this list:", len(randomized))
print("Studies of how the measurement itself behaves:", sum(e[2] == "method" for e in evidence))

In [ ]:
# Checks
assert len(randomized) == 1 and randomized[0][0] == "Marrouche 2022"

An association between a measurement and an outcome does not show that acting on the measurement helps; the
trial that tested it found no difference. And the measurement itself depends on the method and threshold.

## Where to go next

- In part B, lower $T_2$ to 10 ms. What happens to $M_z$ just after the pulse, and why?
- In part D, which TI maximizes the scar-to-blood difference with the illustrative blood T1?
- In part E, add a smooth gain across the wall (coil shading) before normalizing. Which methods suffer most?
- In part F, change the shell from 0–3 mm to 1–4 mm. How much do the numbers move?